# ES4304 — Data Access Accounts

To take part in the course you need accounts with two data providers, to download the satellite data the tutorials use.

**You MUST do this before the course starts.** You cannot process any data without these accounts, and sorting them out during a session wastes your time and everyone else's.

| Provider | Needed for | Register at | Approval |
|---|---|---|---|
| **NASA Earthdata** | PACE (2.1), SWOT (2.3), OSCAR (2.4) | <https://urs.earthdata.nasa.gov/users/new> | Immediate |
| **JAXA P-Tree** | Himawari SST (2.2) | <https://www.eorc.jaxa.jp/ptree/registration_top.html> | **Up to several working days** |

Both are free and quick to fill in. The JAXA one cannot be rushed, though — a person approves it, so register now rather than the night before Tutorial 2.

The more fiddly part is telling the notebooks about your accounts, which is what the rest of this notebook does. **Read each section and edit the code cells carefully.**

## 1. Setup

The same cell appears at the top of every notebook in this course. It installs only what is missing, so it costs nothing when there is nothing to do.

In [ ]:
# --- Setup: run this cell first. Safe to re-run. ---------------------------
# In a Codespace, or a local conda env, everything is already installed and
# this cell does nothing. It installs anything missing, as a safety net.
import importlib.util
import os
import subprocess
import sys

REQUIRED = {                      # import name -> pip name
    "earthaccess": "earthaccess>=0.16",
}

missing = [pip for mod, pip in REQUIRED.items()
           if importlib.util.find_spec(mod) is None]

if missing:
    print("Installing:", ", ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing],
                   check=True)
else:
    print("Environment ready - nothing to install.")

## 2. The `.netrc` file

Both providers are reached over the network with a username and password. Rather than typing them every time, you save them once in a hidden file called **`.netrc`** in your home directory. The `earthaccess` library and Python's `ftplib` both know to look there, so from then on the tutorial notebooks log in without asking you anything.

`.netrc` is a standard Unix file, one line per machine:

```
machine <hostname> login <username> password <password>
```

It lives in your **home directory** (`$HOME`), which is outside this repository, and it survives kernel restarts. In a Codespace it persists for the life of that codespace — if you delete the codespace and create a new one, run this notebook again.

> **The one thing to be careful about.** You are about to type your passwords into code cells. The `.netrc` file itself is fine, but a **saved notebook keeps whatever you typed into a cell** — and this notebook is in a Git repository. Section 4 puts the placeholders back for you when you are done; do not skip it.

### NASA Earthdata

Getting data from any NASA Distributed Active Archive Center (DAAC) requires a free Earthdata account. Register — or log in, if you already have one — at the Earthdata User Registration Service: <https://urs.earthdata.nasa.gov/home>. You are sent to this same site whenever you download NASA data through a browser.

Substitute **your** Earthdata username for `myUsername` and **your** password for `myPassword` below, then run the cell. You can equally run the same command in a terminal.

In [ ]:
# if you don't substitute your own details here, this won't work
!echo "machine urs.earthdata.nasa.gov login myUsername password myPassword" > $HOME/.netrc

### JAXA P-Tree

P-Tree is JAXA's Himawari data portal. When your registration is approved you receive an **FTP username and password** — these are *separate from your P-Tree website login*, which is the single most common thing to get wrong here. The FTP username is usually your email address with the `@` replaced by an underscore.

Details and FAQ: <https://www.eorc.jaxa.jp/ptree/faq.html>

Substitute your P-Tree **FTP** credentials below and run it. Note the `>>` rather than `>` — that appends a second line instead of overwriting the first.

**If your approval has not arrived yet, skip this cell** and come back to it. Nothing before Tutorial 2.2 needs it.

In [ ]:
# again, if you don't edit this to use your own details, this will not work
!echo "machine ftp.ptree.jaxa.jp login myEmail_example.com password myPtreePassword" >> $HOME/.netrc

### Protect the file

`.netrc` now holds your personal passwords, so change its permissions so that nobody else can read it. `600` means readable by you and nobody else. Some programs check this and refuse to run until you do it.

In [ ]:
!chmod 600 $HOME/.netrc

## 3. Check it worked

First, look at what is in the file. This prints the hostnames and usernames, and masks the passwords — so that if you save the notebook with this output in it, nothing leaks.

In [ ]:
import netrc

nrc = netrc.netrc()

for machine in sorted(nrc.hosts):
    login, _, password = nrc.hosts[machine]
    print(f"{machine:30s} login={login!r:30s} password={'*' * len(password)}")

### NASA Earthdata

`earthaccess.login(strategy="netrc")` reads the file and logs in against Earthdata. **This route checks your credentials immediately** — a wrong password raises here, rather than failing later in the middle of a tutorial.

Then we download one real file, about 26 MB, and delete it again. Logging in proves the password is right; downloading proves the account can actually get data.

In [ ]:
import shutil

import earthaccess

auth = earthaccess.login(strategy="netrc")

# `authenticated` is the flag that actually means "logged in". `username` is
# only filled in when the login used a username and password - a token login
# leaves it as None, so printing it alone is misleading.
print("Authenticated:", auth.authenticated)
print("Logged in to Earthdata as:", auth.username or "(token login - no username)")

if auth.username is None:
    print("\nNo username means earthaccess did not use your ~/.netrc entry - it")
    print("was already authenticated with an EARTHDATA_TOKEN. Downloads will")
    print("work, but the credentials you saved above have NOT been checked.")
    print("Run `echo $EARTHDATA_TOKEN` in a terminal; if it prints anything,")
    print("remove it from your Codespaces secrets, then restart the kernel and")
    print("re-run this notebook to test the .netrc credentials themselves.")

results = earthaccess.search_data(
    short_name="PACE_OCI_L3M_BGC",     # chlorophyll lives in the biogeochemistry suite
    version="3.2",
    granule_name="*MO*0p1*",           # monthly, 0.1 degree
    count=1,
)

if not results:
    print("\nSearch found nothing. That is not an account problem - a retired")
    print("short_name returns an empty list rather than an error. See S01.")
else:
    print("Found:", results[0].data_links()[0].split("/")[-1])

    TEST_DIR = os.path.join(os.getcwd(), "netrc_check")
    try:
        files = earthaccess.download(results, TEST_DIR)
        print("Downloaded:", os.path.basename(files[0]))
        print("\nNASA Earthdata access works.")
    finally:
        shutil.rmtree(TEST_DIR, ignore_errors=True)

### JAXA P-Tree

`ftplib` does not read `.netrc` by itself, so we use the standard library's `netrc` module to pull the credentials out and hand them over. Every notebook that needs P-Tree does exactly this, which is why you never type the password again.

Skip this cell if your approval has not arrived.

In [ ]:
import netrc
from ftplib import FTP

FTP_SERVER = "ftp.ptree.jaxa.jp"

ftp_user, _, ftp_password = netrc.netrc().authenticators(FTP_SERVER)

with FTP(FTP_SERVER) as ftp:
    ftp.login(ftp_user, ftp_password)
    print("Connected. Top-level directories:", ftp.nlst()[:5])
    print("\nJAXA P-Tree access works.")

## 4. Tidy up before you commit

Your `~/.netrc` is in your home directory, not in this repository, so it is never committed and keeps working after this.

**The code cells above are in the repository.** Run the cell below: it rewrites this notebook file with the placeholder text back in place and clears the outputs, so your passwords are not saved into it.

In [ ]:
# Restores the placeholders in this notebook's own file. Your ~/.netrc is
# untouched and keeps working.
import json
import re

NB_PATH = "0.2.1_Account_Check.ipynb"

# .*? and .* rather than \S+, so a username or password containing a space is
# still scrubbed rather than silently left behind.
PATTERNS = [
    (r'(machine urs\.earthdata\.nasa\.gov login ).*?( password ).*(")',
     r'\1myUsername\2myPassword\3'),
    (r'(machine ftp\.ptree\.jaxa\.jp login ).*?( password ).*(")',
     r'\1myEmail_example.com\2myPtreePassword\3'),
]

with open(NB_PATH) as fh:
    nb = json.load(fh)

changed = 0
for cell in nb["cells"]:
    for i, line in enumerate(cell["source"]):
        for pattern, replacement in PATTERNS:
            new = re.sub(pattern, replacement, line)
            if new != line:
                cell["source"][i] = new
                line = new
                changed += 1
    if cell["cell_type"] == "code":
        cell["outputs"] = []
        cell["execution_count"] = None

with open(NB_PATH, "w") as fh:
    json.dump(nb, fh, indent=1, ensure_ascii=False)
    fh.write("\n")

if changed:
    print(f"Placeholders restored on {changed} line(s); outputs cleared.")
    print("Close and reopen this notebook so the editor picks up the change.")
else:
    print("Nothing was replaced. Either the placeholders are already back, or")
    print("the write cells were edited into a shape this does not recognise -")
    print("check them by eye before committing.")

## If something failed

- **`FileNotFoundError`, or "No .netrc found".** The write cell did not run, or you are in a brand-new codespace. Run section 2 again.
- **`LoginAttemptFailure`.** Earthdata rejected the username or password. Check them by logging in at <https://urs.earthdata.nasa.gov>, then re-run the Earthdata write cell — it uses `>` and rewrites the whole file, so run the JAXA cell again after it.
- **"Logged in to Earthdata as: None", or "(token login - no username)".** You are authenticated, but not with the `.netrc` entry you just wrote — an `EARTHDATA_TOKEN` in the environment got there first, and `earthaccess` skips the `.netrc` lookup once it is already logged in. Downloads work, but this notebook has not verified your username and password. Run `echo $EARTHDATA_TOKEN` in a terminal; if it prints anything, remove it from your Codespaces secrets (**Settings → Codespaces → Repository secrets**), then rebuild or restart the kernel and re-run.
- **`530 Login incorrect`** from JAXA. You used your website login rather than the FTP credentials, or your registration is not approved yet.
- **`TypeError: cannot unpack non-sequence NoneType`** in the P-Tree cell. There is no `ftp.ptree.jaxa.jp` line in your `.netrc` — run the JAXA write cell.
- **A password with a `"` in it.** The write cells wrap the line in double quotes, so a double quote in your password breaks the command. Change your password, or write the `.netrc` line by hand in a terminal.
- Anything else: [S01 Troubleshooting](../S01_Troubleshooting/README.md).

## Next

[Tutorial 2 overview](../2.0_Tutorial_2_Overview_and_Assignment/README.md)